<a href="https://colab.research.google.com/github/NKVRK/Resume-Analyzer-using-FastAPI/blob/main/notebooks/en/code_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q faiss-cpu fastembed inflection tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.3/105.3 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 121.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.8/324.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 8.8 MB/s eta 0:00:00


In [2]:
import os, json, re
from pathlib import Path
!unzip -o /content/Resume-Analyzer-using-FastAPI-main.zip -d /content

BASE_PATH = "/content/Resume-Analyzer-using-FastAPI-main"

BACKEND_ROOT = Path(BASE_PATH) / "backend"
FRONTEND_ROOT = Path(BASE_PATH) / "frontend"

EXCLUDE_FOLDERS = {"venv", ".venv", "node_modules", "__pycache__", ".git", "embeddings", "dist", "build", ".next"}

BACKEND_EXTS = {".py"}
FRONTEND_EXTS = {".js", ".jsx", ".ts", ".tsx"}

EMB_DIR = os.path.join(BASE_PATH, "embeddings")
os.makedirs(EMB_DIR, exist_ok=True)

TEXT_INDEX_PATH = os.path.join(EMB_DIR, "text_index.faiss")
TEXT_META_PATH  = os.path.join(EMB_DIR, "text_meta.jsonl")

BACKEND_INDEX_PATH = os.path.join(EMB_DIR, "backend_code_index.faiss")
BACKEND_META_PATH  = os.path.join(EMB_DIR, "backend_code_meta.jsonl")

FRONTEND_INDEX_PATH = os.path.join(EMB_DIR, "frontend_code_index.faiss")
FRONTEND_META_PATH  = os.path.join(EMB_DIR, "frontend_code_meta.jsonl")

print("Base Repo:", BASE_PATH)
print("Embeddings Location:", EMB_DIR)


Base Repo: /root/Downloads/Resume-Analyzer-using-FastAPI
Embeddings Location: /root/Downloads/Resume-Analyzer-using-FastAPI/embeddings


In [3]:
def is_excluded(p: Path):
    return any(part in EXCLUDE_FOLDERS for part in p.parts)

def list_files(root, exts):
    if not root.exists(): return []
    files = []
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in exts and not is_excluded(p):
            files.append(str(p))
    return files

backend_files = list_files(BACKEND_ROOT, BACKEND_EXTS)
frontend_files = list_files(FRONTEND_ROOT, FRONTEND_EXTS)

print(f"Backend files: {len(backend_files)}")
print(f"Frontend files: {len(frontend_files)}")


Backend files: 0
Frontend files: 0


In [4]:
from tqdm import tqdm

def read_file(path):
    try: return Path(path).read_text(encoding="utf-8", errors="ignore")
    except: return ""

def chunk_lines(text, max_lines=40):
    lines = text.splitlines()
    return ["\n".join(lines[i:i+max_lines]) for i in range(0, len(lines), max_lines) if "".join(lines[i:i+max_lines]).strip()]

def to_structure(file_path, snippet, line_from, line_to):
    rel = os.path.relpath(file_path, BASE_PATH)
    module = Path(rel).parent.as_posix().replace("/", "_") or "root"

    return {
        "signature": snippet.splitlines()[0][:200] if snippet else "",
        "context": {
            "module": module,
            "file_path": rel,
            "file_name": Path(file_path).name,
            "snippet": snippet,
        },
        "line_from": line_from,
        "line_to": line_to,
    }

def build_structures(files):
    structs = []
    for fp in tqdm(files, desc="Chunking files"):
        txt = read_file(fp)
        start = 1
        for c in chunk_lines(txt, 40):
            line_count = c.count("\n") + 1
            structs.append(to_structure(fp, c, start, start + line_count - 1))
            start += line_count
    return structs

backend_structures = build_structures(backend_files)
frontend_structures = build_structures(frontend_files)

print("Backend chunks:", len(backend_structures))
print("Frontend chunks:", len(frontend_structures))


Chunking files: 0it [00:00, ?it/s]
Chunking files: 0it [00:00, ?it/s]

Backend chunks: 0
Frontend chunks: 0


In [5]:
import inflection

def textify(s):
    sig = inflection.humanize(inflection.underscore(s["signature"])) if s["signature"] else ""
    return f"Code section defined in module {s['context']['module']} with signature {sig}".strip()

text_structures = backend_structures + frontend_structures
text_representations = [textify(s) for s in text_structures]
print("Total text entries:", len(text_representations))


Total text entries: 0


In [6]:
from fastembed import TextEmbedding

BATCH_SIZE = 8

nlp_model  = TextEmbedding("sentence-transformers/all-MiniLM-L6-v2")
code_model = TextEmbedding("jinaai/jina-embeddings-v2-base-code")

print("✅ Models loaded")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/493 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

onnx/model.onnx:   0%|          | 0.00/642M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Models loaded


In [7]:
import faiss
import numpy as np

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r)+"\n")

def read_jsonl(path):
    return [json.loads(l) for l in open(path, "r", encoding="utf-8")]

def new_index(dim):
    return faiss.IndexFlatIP(dim)


In [8]:
def ensure_text_index():
    if os.path.exists(TEXT_INDEX_PATH) and os.path.exists(TEXT_META_PATH):
        print("✅ Loaded TEXT index")
        return faiss.read_index(TEXT_INDEX_PATH), read_jsonl(TEXT_META_PATH)

    print("Building TEXT index...")
    idx = None
    meta = []
    buf = []

    gen = nlp_model.embed(text_representations, batch_size=BATCH_SIZE)
    for emb, s in tqdm(zip(gen, text_structures), total=len(text_structures)):
        buf.append(emb)
        meta.append(s)
        if len(buf) >= BATCH_SIZE:
            vecs = np.array(buf, dtype="float32")
            faiss.normalize_L2(vecs)
            if idx is None: idx = new_index(vecs.shape[1])
            idx.add(vecs); buf=[]
    if buf:
        vecs = np.array(buf, dtype="float32")
        faiss.normalize_L2(vecs)
        if idx is None: idx = new_index(vecs.shape[1])
        idx.add(vecs)

    faiss.write_index(idx, TEXT_INDEX_PATH)
    write_jsonl(TEXT_META_PATH, meta)
    print("✅ TEXT index built")
    return idx, meta

text_index, text_meta = ensure_text_index()


Building TEXT index...


0it [00:00, ?it/s]

✅ TEXT index built


In [9]:
def ensure_backend_index():
    if os.path.exists(BACKEND_INDEX_PATH) and os.path.exists(BACKEND_META_PATH):
        print("✅ Loaded BACKEND code index")
        return faiss.read_index(BACKEND_INDEX_PATH), read_jsonl(BACKEND_META_PATH)

    print("Building BACKEND code index...")
    idx, meta, buf = None, [], []
    gen = code_model.embed([s["context"]["snippet"] for s in backend_structures], batch_size=BATCH_SIZE)

    for emb, s in tqdm(zip(gen, backend_structures), total=len(backend_structures)):
        buf.append(emb); meta.append(s)
        if len(buf) >= BATCH_SIZE:
            vecs = np.array(buf, dtype="float32")
            faiss.normalize_L2(vecs)
            if idx is None: idx = new_index(vecs.shape[1])
            idx.add(vecs); buf=[]
    if buf:
        vecs = np.array(buf, dtype="float32")
        faiss.normalize_L2(vecs)
        if idx is None: idx = new_index(vecs.shape[1])
        idx.add(vecs)

    faiss.write_index(idx, BACKEND_INDEX_PATH)
    write_jsonl(BACKEND_META_PATH, meta)
    print("✅ BACKEND code index built")
    return idx, meta

backend_index, backend_meta = ensure_backend_index()


Building BACKEND code index...


0it [00:00, ?it/s]

✅ BACKEND code index built


In [10]:
def ensure_frontend_index():
    if os.path.exists(FRONTEND_INDEX_PATH) and os.path.exists(FRONTEND_META_PATH):
        print("✅ Loaded FRONTEND code index")
        return faiss.read_index(FRONTEND_INDEX_PATH), read_jsonl(FRONTEND_META_PATH)

    print("Building FRONTEND code index...")
    idx, meta, buf = None, [], []
    gen = code_model.embed([s["context"]["snippet"] for s in frontend_structures], batch_size=BATCH_SIZE)

    for emb, s in tqdm(zip(gen, frontend_structures), total=len(frontend_structures)):
        buf.append(emb); meta.append(s)
        if len(buf) >= BATCH_SIZE:
            vecs = np.array(buf, dtype="float32")
            faiss.normalize_L2(vecs)
            if idx is None: idx = new_index(vecs.shape[1])
            idx.add(vecs); buf=[]
    if buf:
        vecs = np.array(buf, dtype="float32")
        faiss.normalize_L2(vecs)
        if idx is None: idx = new_index(vecs.shape[1])
        idx.add(vecs)

    faiss.write_index(idx, FRONTEND_INDEX_PATH)
    write_jsonl(FRONTEND_META_PATH, meta)
    print("✅ FRONTEND code index built")
    return idx, meta

frontend_index, frontend_meta = ensure_frontend_index()


Building FRONTEND code index...


0it [00:00, ?it/s]

✅ FRONTEND code index built


In [11]:
def emb_text(q):
    v = next(nlp_model.query_embed(q))
    v = np.array([v], dtype="float32")
    faiss.normalize_L2(v)
    return v

def emb_code(q):
    v = next(code_model.query_embed(q))
    v = np.array([v], dtype="float32")
    faiss.normalize_L2(v)
    return v

def run_search(index, meta, qvec, k=5):
    D, I = index.search(qvec, k)
    results = []
    for score, idx in zip(D[0], I[0]):
        if idx < 0: continue
        s = meta[idx]
        results.append((score, s["context"]["file_path"], s["context"]["snippet"]))
    return results

def show(results):
    for i,(score,fp,snip) in enumerate(results,1):
        print(f"\n#{i} score={score:.4f} :: {fp}\n{'-'*60}\n{snip[:800]}")


In [12]:
def search_backend_code(query):
    show(run_search(backend_index, backend_meta, emb_code(query)))

def search_frontend_code(query):
    show(run_search(frontend_index, frontend_meta, emb_code(query)))

def search_text(query):
    show(run_search(text_index, text_meta, emb_text(query)))


In [14]:
!ls -R ~/Downloads/Resume-Analyzer-using-FastAPI


/root/Downloads/Resume-Analyzer-using-FastAPI:
embeddings

/root/Downloads/Resume-Analyzer-using-FastAPI/embeddings:
backend_code_index.faiss  frontend_code_index.faiss  text_index.faiss
backend_code_meta.jsonl   frontend_code_meta.jsonl   text_meta.jsonl


In [13]:
search_backend_code("extract skills from resume")
search_frontend_code("handle form submit")
search_text("where does resume parsing happen")


AttributeError: 'NoneType' object has no attribute 'search'